In [ ]:
!nvidia-smi -l 1

In [ ]:
%pip install transformer_lens
%pip install circuitsvis

In [ ]:
# import ipykernel
# import subprocess
# import platform

# def spawn_matlab_style_console():
#     conn_file = ipykernel.connect.get_connection_file()
#     cmd = f"jupyter console --existing {conn_file}"
#     sys_os = platform.system()
    
#     try:
#         if sys_os == 'Windows':
#             # Opens a new Command Prompt
#             subprocess.Popen(f'start cmd /k {cmd}', shell=True)
#         elif sys_os == 'Darwin':
#             # Opens a new macOS Terminal
#             subprocess.Popen(['osascript', '-e', f'tell application "Terminal" to do script "{cmd}"'])
#         elif sys_os == 'Linux':
#             # Attempts to open default Linux terminal
#             subprocess.Popen(['x-terminal-emulator', '-e', cmd])
#         print("✅ MATLAB-style console launched in a new window!")
#     except Exception as e:
#         print(f"Failed to launch terminal: {e}")

# spawn_matlab_style_console()

In [ ]:
import ipykernel
from IPython.display import display, HTML

# Get the exact absolute path of the current active kernel
conn_file = ipykernel.connect.get_connection_file()
cmd = f"jupyter console --existing {conn_file}"

# Generate a UI button to copy the command
html = f"""
<div style="background: #1e1e1e; color: #d4d4d4; padding: 12px; border-radius: 4px; font-family: monospace; border: 1px solid #333;">
    <span id="jupyter_cmd">{cmd}</span><br><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('jupyter_cmd').innerText);" 
            style="background: #007acc; color: white; border: none; padding: 6px 1144/62px; border-radius: 3px; cursor: pointer;">
        Copy to VSCode Terminal
    </button>
</div>
"""
display(HTML(html))

In [ ]:
import sys
sys.path.append('/home/galk/LanguageDynamics/src') 

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torch.distributions.kl import kl_divergence
from torch.distributions.categorical import Categorical
from torch.distributions.multivariate_normal import MultivariateNormal
# from datasets import load_dataset
from datetime import datetime

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# import interpretability stuff
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix
import circuitsvis as cv
import einops

from entmax import entmax15, sparsemax

from importlib import reload

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
## Activation Patching Functions

from functools import partial

def patch_head_hook(corrupted_hook, hook, clean_cache, head_idx, position=None, knockout=False):
    """
    The hook function that surgically overwrites the corrupted activations 
    with the clean activations for a specific head and position.
    
    corrupted_hook shape: [batch, sequence_length, n_heads, d_head]
    """
    # Fetch the clean activations from the cache
    clean_hook = clean_cache[hook.name]
    if knockout:
        clean_hook = torch.zeros_like(clean_hook)
    
    if position is None:
        # Wide Patching: Patch this head at ALL sequence positions
        corrupted_hook[:, :, head_idx, :] = clean_hook[:, :, head_idx, :]
    else:
        # Position-Specific Patching: Patch this head at a SINGLE position
        corrupted_hook[:, position, head_idx, :] = clean_hook[:, position, head_idx, :]
        
    return corrupted_hook


def run_attention_patching(
    model: HookedTransformer,
    clean_prompt: str,
    corrupt_prompt: str,
    clean_answer: str,
    corrupt_answer: str,
    layer: int,
    head_idx: int,
    position: int = None
):
    """
    Runs the full causal tracing experiment and returns logits and the recovery score.
    """
    # 1. Ensure prompts are tokenized to the same length for clean position mapping
    clean_tokens = model.to_tokens(clean_prompt)
    corrupt_tokens = model.to_tokens(corrupt_prompt)
    
    if clean_tokens.shape[1] != corrupt_tokens.shape[1]:
        print("Warning: Prompts have different token lengths. Patching by position may behave unexpectedly.")

    # Get token IDs for the expected answers to calculate our metric
    clean_answer_id = model.to_single_token(clean_answer)
    corrupt_answer_id = model.to_single_token(corrupt_answer)

    # 2. Run the CLEAN prompt and cache all activations
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)

    # 3. Run the CORRUPT prompt to get baseline corrupted logits
    corrupt_logits = model(corrupt_tokens)

    # 4. Run the PATCHED prompt
    # We hook into the 'z' activation (the mixed values right before the W_O projection)
    hook_name = f"blocks.{layer}.attn.hook_z"
    
    # Use functools.partial to freeze our specific arguments into the hook function
    hook_fn = partial(
        patch_head_hook, 
        clean_cache=clean_cache, 
        head_idx=head_idx, 
        position=position
    )
    
    patched_logits = model.run_with_hooks(
        corrupt_tokens,
        fwd_hooks=[(hook_name, hook_fn)]
    )

    # 5. Calculate Metrics (Logit Difference at the final token position)
    def get_logit_diff(logits):
        # Look at the final token's prediction [batch_idx=0, pos_idx=-1]
        final_logits = logits[0, -1, :]
        return (final_logits[clean_answer_id] - final_logits[corrupt_answer_id]).item()

    clean_diff = get_logit_diff(clean_logits)
    corrupt_diff = get_logit_diff(corrupt_logits)
    patched_diff = get_logit_diff(patched_logits)

    # Recovery Score: 0% means it acts like the corrupted model, 100% means it acts like the clean model
    recovery_score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)

    return {
        "clean_logits": clean_logits,
        "corrupt_logits": corrupt_logits,
        "patched_logits": patched_logits,
        "clean_diff": clean_diff,
        "corrupt_diff": corrupt_diff,
        "patched_diff": patched_diff,
        "recovery_score": recovery_score
    }

def get_logit_diff(logits, clean_id, corrupt_id):
    """Calculates Logit(Clean) - Logit(Corrupt) for the final token."""
    final_logits = logits[0, -1, :]
    return (final_logits[clean_id] - final_logits[corrupt_id]).item()

def get_probability(logits, target_id):
    """Calculates the softmax probability of a specific target token."""
    final_logits = logits[0, -1, :]
    probs = F.softmax(final_logits, dim=-1)
    return probs[target_id].item()

def zero_head_hook(z, hook, head_idx, min_position=0):
    """
    An ablation hook that completely zeroes out a specific head's output.
    Because this runs during generation with KV caching, 'z' might be the 
    activations for the whole prompt, or just the single newest token.
    Using `:, :, head_idx, :` handles both cases perfectly.
    
    z shape: [batch, sequence_length, n_heads, d_head]
    """
    # Set all activations for this specific head to 0
    if z.shape[1] > min_position:
        z[:, min_position:, head_idx, :] = 0.0
    else:
        z[:, :, head_idx, :] = 0.0
    return z

def remove_pos_embed_hook(pos_embed, hook):
    return torch.zeros_like(pos_embed)

def generate_with_knockout(model, prompt: str, layer: int, head_idx: int, max_tokens: int = 20, knockout_positional=False, do_sample=False, top_p=0.9, prepend_bos=True):
    """
    Generates text while knocking out a specific head, ensuring safe hook removal.
    """
    hook_name = f"blocks.{layer}.attn.hook_z"
    hook_fn = partial(zero_head_hook, head_idx=head_idx)

    hooks = [(hook_name, hook_fn)]

    if knockout_positional:
        positional_hook_name = "hook_pos_embed"
        positional_hook_fn = remove_pos_embed_hook
        hooks += [(positional_hook_name, positional_hook_fn)]
    
    print(f"--- Intervening: Knocking out L{layer}H{head_idx} ---")
    
    # The context manager ensures hooks exist ONLY inside this indented block
    with model.hooks(fwd_hooks=hooks):
        
        # You can use the standard generate method exactly as normal
        patched_output = model.generate(
            prompt,
            max_new_tokens=max_tokens,
            do_sample=do_sample,
            temperature=1.0,
            top_p=top_p,
            prepend_bos=prepend_bos 
        )
    
    return patched_output

In [ ]:
## Optimization and Black Box Functions
def pi_to_P_one_layer(pi: torch.Tensor, heads: list, model: HookedTransformer, chunk_size=128, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        # convert the chunk to sparse representation
        chunk_sparse = chunk_probs.to_sparse()
        return chunk_sparse.indices(), chunk_sparse.values()

    # Lists to accumulate sparse tensor components
    all_indices = []
    all_values = []
    # compute queries and outputs in chunks
    for i in tqdm(range(0, model.cfg.d_vocab, chunk_size), desc="Chunking Sparse P:"):
        q = einops.einsum(attention_resid_pre[i:i+chunk_size], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
        q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
        E_chunk = E[i:i+chunk_size]
        
        # Apply checkpointing over chunks
        chunk_indices, chunk_values = checkpoint(
            compute_P_chunk,
            pi, q, E_chunk,
            use_reentrant=False
        )
        
        # Clone indices to avoid modifying the original view, then apply row offset
        indices = chunk_indices.clone()
        indices[0] += i # 'i' is the current row offset in the d_vocab loop

        all_indices.append(indices)
        all_values.append(chunk_values)

        # logit_matrix_chunks.append(chunk_logits_matrix)

    ## Build final sparse matrix
    # Concatenate all indices and values
    full_indices = torch.cat(all_indices, dim=1)
    full_values = torch.cat(all_values, dim=0)

    # Construct the final full d_vocab x d_vocab sparse transition matrix
    full_sparse_P = torch.sparse_coo_tensor(
        full_indices, 
        full_values, 
        size=(model.cfg.d_vocab, model.cfg.d_vocab)
    )

    # full_logits_matrix = torch.cat(logit_matrix_chunks)
    return full_sparse_P

def pi_to_pi_P_one_layer(pi: torch.Tensor, heads: list, model: HookedTransformer, chunk_size=512, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        return chunk_probs # shape: [chunk_size, vocab]
    
    def compute_pi_P_chunk(pi, q_chunk, E_chunk, pi_chunk):
        # compute the dense chunk of the P matrix
        chunk_probs = compute_P_chunk(pi, q_chunk, E_chunk)
        return pi_chunk.view(1, -1) @ chunk_probs

    # Lists to accumulate the vector-matrix product
    pi_P_accum = torch.zeros_like(pi)

    # compute queries and outputs in chunks
    for i in tqdm(range(0, model.cfg.d_vocab, chunk_size), desc="Chunking pi_P"):
        q = einops.einsum(attention_resid_pre[i:i+chunk_size], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
        q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
        E_chunk = E[i:i+chunk_size]
        pi_chunk = pi[i:i+chunk_size]
        
        # Apply checkpointing over chunks
        pi_P_chunk = checkpoint(
            compute_pi_P_chunk,
            pi, q, E_chunk, pi_chunk,
            use_reentrant=False
        )

        pi_P_accum += pi_P_chunk.squeeze(0)

    return pi_P_accum

def pi_to_pi_P_one_layer_topk(pi: torch.Tensor, heads: list, model: HookedTransformer, top_tokens, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        return chunk_probs # shape: [chunk_size, vocab]
    
    def compute_pi_P_chunk(pi, q_chunk, E_chunk, pi_chunk):
        # compute the dense chunk of the P matrix
        chunk_probs = compute_P_chunk(pi, q_chunk, E_chunk)
        return pi_chunk.view(1, -1) @ chunk_probs

    # Lists to accumulate the vector-matrix product
    # pi_P_accum = torch.zeros_like(pi)

    # compute queries and outputs in chunks
    q = einops.einsum(attention_resid_pre[top_tokens], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
    q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
    E_chunk = E[top_tokens]
    pi_chunk = pi[top_tokens]
    
    # Apply checkpointing over chunks
    # pi_P_chunk = checkpoint(
    #     compute_pi_P_chunk,
    #     pi, q, E_chunk, pi_chunk,
    #     use_reentrant=False
    # )

    pi_P = compute_pi_P_chunk(pi, q, E_chunk, pi_chunk).squeeze(0)

    # pi_P_accum += pi_P_chunk.squeeze(0)

    return pi_P

def pi_from_context(tokens: torch.Tensor, vocab_size=48262):
    pi = torch.zeros(vocab_size, device=tokens.device)
    n_context = tokens.numel()

    values, counts = torch.unique(tokens, return_counts=True)
    pi[values] = counts.float()
    pi = pi / n_context
    return pi

def context_from_pi(pi: torch.Tensor, N: int):
    # pi[1] = 0 # zero out BOS probability
    threshold = 1/(2*N)
    pi[pi < threshold] = 0
    tokens = torch.multinomial(pi, num_samples=N, replacement=True)
    tokens_BOS_prepended = torch.cat((torch.tensor([1]), tokens), dim=0) 
    return tokens

def pi_t_from_context(tokens: torch.Tensor, vocab_size=48262):
    n_context = tokens.numel()

    pi_t = [pi_from_context(tokens[:t]).view(1, -1) for t in range(1,n_context+1)]
    return torch.cat(pi_t, dim=0)

In [ ]:
# Load model
# device = utils.get_device()
device = 'cuda:5'
model_name = "attn-only-1l"
model = HookedTransformer.from_pretrained(model_name, device=device)

model.cfg.use_attn_result = True # make the model use the inefficient OV computation
# torch.set_grad_enabled(False) # turn off automatic differentiation

In [ ]:
# generate
prompt = "I walked out of the machine and into the forest. You"
noise = "So many millions of tokens of Arbitrary Noise that interrupt the prompt repeatedly that seem unrelated.\nWill this matter to the model at all?\nI believe that it should, because the doctor said so."
input_tokens = model.to_tokens(prompt)

generated = model.generate(
    prompt,
    max_new_tokens=100,
    do_sample=False,
    temperature=1.0,
    top_p=1.0,
    prepend_bos=True
)

print(generated)

In [ ]:
## Generation with Knockout
# prompt = " Joe" * 10
prompt = "Joe robbed a bank on Monday. On Tuesday"
# prompt = bla
# prompt = torch.tensor([[4329] * 100]).to(device)
max_new_tokens = 1000

do_sample = True
top_p = 0.9
prepend_bos = True

print("Normal Generation:")
clean_output = model.generate(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample, top_p=top_p, prepend_bos=prepend_bos)
print(f"Result: {clean_output}\n")

# 2. Generate with our temporary knockout
target_layer = 0
target_head = []
knockout_positional = True

patched_output = generate_with_knockout(
    model, 
    prompt, 
    layer=target_layer, 
    head_idx=target_head, 
    max_tokens=max_new_tokens,
    knockout_positional=knockout_positional,
    do_sample=do_sample,
    top_p=top_p,
    prepend_bos=prepend_bos
)

print(f"Result: {patched_output}\n")


In [ ]:
# Run with cache
# prompt = " Jack did football when movie." * 170
# prompt = ' yes' * 1000
# input_tokens = model.to_tokens(prompt)
# input_tokens = torch.tensor([[1] + [18462] * 1000])
# input_tokens = model.to_tokens(generated)
input_tokens = model.to_tokens(patched_output)
# print(input_tokens.device)
layer = 0
head_idx = [0,2,3,4,5,6,7]
min_position = 0
knockout_positional = False


hook_name = f"blocks.{layer}.attn.hook_result"
hook_fnc = partial(zero_head_hook, head_idx=head_idx, min_position=min_position)
hooks = [(hook_name, hook_fnc)]

if knockout_positional:
    positional_hook_name = "hook_pos_embed"
    positional_hook_fn = remove_pos_embed_hook
    hooks += [(positional_hook_name, positional_hook_fn)]

with model.hooks(fwd_hooks=hooks):
    logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True)
    
probs = F.softmax(logits, dim=-1).cpu().numpy()

In [ ]:
### DLA computation ###

# Compute DLA by projecting the head results through the unembedding matrix (W_U)
# W_U shape: [d_model, vocab_size]
# Output shape: [seq_len, num_heads, vocab_size]
dla_0 = einops.einsum(
    cache["blocks.0.attn.hook_result"] - cache["blocks.0.attn.hook_result"].mean(dim=-1, keepdim=True), 
    model.W_U, 
    "seq_len num_heads d_model, d_model vocab_size -> seq_len num_heads vocab_size"
)
if model.cfg.n_layers > 1:
    dla_1 = einops.einsum(
        cache["blocks.1.attn.hook_result"] - cache["blocks.1.attn.hook_result"].mean(dim=-1, keepdim=True), 
        model.W_U, 
        "seq_len num_heads d_model, d_model vocab_size -> seq_len num_heads vocab_size"
    )
else:
    dla_1 = torch.zeros_like(dla_0)
dla_x0 = einops.einsum(
    cache["blocks.0.hook_resid_pre"] - cache["blocks.0.hook_resid_pre"].mean(dim=-1, keepdim=True), 
    model.W_U, 
    "seq_len d_model, d_model vocab_size -> seq_len vocab_size"
)
# ln_bias_dla = einops.einsum(
#     model.ln_final.b,
#     model.W_U,
#     "d_model, d_model vocab_size -> vocab_size"
# )
b_O_0_dla = (model.blocks[0].attn.b_O - model.blocks[0].attn.b_O.mean(dim=-1, keepdim=True)) @ model.W_U
unembed_bias_dla = model.b_U 
# total_bias_logits = ln_bias_dla + unembed_bias_dla

# ---Apply Final LayerNorm Scaling ---
# True DLA often divides by the scaling factor of the final LayerNorm to reflect 
# the exact contribution to the final logits. Uncomment if you need strict exactness.
ln_scale = cache["ln_final.hook_scale"] # Shape: [seq_len, 1]
# We unsqueeze to broadcast across the num_heads dimension: [seq_len, 1, 1]
dla_0 = dla_0 / ln_scale.unsqueeze(1)
dla_1 = dla_1 / ln_scale.unsqueeze(1)
dla_x0 = dla_x0 / ln_scale
b_O_0_dla = b_O_0_dla / ln_scale

dla = torch.concat([dla_x0.unsqueeze(1), unembed_bias_dla.unsqueeze(0).repeat(dla_0.shape[0], 1).unsqueeze(1), dla_0, b_O_0_dla.unsqueeze(1), dla_1], dim=1)

print(f"DLA tensor shape: {dla.shape}")

In [ ]:
prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]


prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
# prompt_dla = prompt_dla - prompt_dla[:, 1:2]
max_dla = torch.max(torch.abs(prompt_dla)).item()

# Plot as heatmap
# plt.figure(figsize=(20, 8))
# plt.imshow((prompt_dla).cpu().numpy().T, cmap='coolwarm', aspect='auto', vmin=-max_dla, vmax=max_dla)
# plt.colorbar(label='DLA')
# plt.xlabel('Position')
# plt.ylabel('Head')
# plt.yticks(range(18), ["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(2) for head in range(8)])
# plt.title(f'DLA of all heads')
# plt.tight_layout()

plt.figure(figsize=(15,8))
plt.plot(prompt_dla[:,:11].cpu().numpy(), "--o", label=["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(1) for head in range(8)] + [f"$b_(O0)$"])
plt.plot(prompt_dla[:,11:].cpu().numpy(), "--x", label=[f"L{layer}H{head}" for layer in range(1,2) for head in range(8)])
plt.xlabel("Position")
plt.ylabel("DLA")
plt.grid()
plt.legend()

plt.show()

In [ ]:
### Plot prompt logits, max logits and 2-nd max logits

prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]

prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
# prompt_dla = prompt_dla - prompt_dla[:, 1:2]
max_dla = torch.max(torch.abs(prompt_dla)).item()

prompt_logits = logits[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits = torch.max(logits[0, :-1], dim=-1).values
prompt_max_2_logits = torch.topk(logits[0, :-1], k=2, dim=-1).values[:, 1]

plt.figure(figsize=(10,8))
plt.plot(prompt_logits.cpu().numpy())
# plt.plot(prompt_dla.sum(dim=-1).cpu().numpy())
plt.plot(prompt_max_logits.cpu().numpy(), '--r')
plt.plot(prompt_max_2_logits.cpu().numpy(), '--b')
plt.grid()
plt.title(f"Chosen and Maximal Logits")
plt.xlabel("Position")
plt.ylabel("Logit")
plt.figure(figsize=(10,8))
# plt.plot(prompt_logits[350:470].cpu().numpy())
plt.plot((prompt_max_logits-prompt_max_2_logits).cpu().numpy(), '.')
# plt.stem(.cpu().numpy(), 'b.')
plt.grid()
plt.title(f"Top-1 to Top-2 Logit Diff")
plt.xlabel("Position")
plt.ylabel("Logit Diff")

plt.show()

In [ ]:
### DLA of Repetitive Pattern
jump = 1
prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]


# pick out the chosen tokens
prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
prompt_logits = logits[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits = torch.max(logits[0, :-1], dim=-1).values

prompt_dla_matrix = prompt_dla.reshape(-1, jump, 18)
prompt_logits_matrix = prompt_logits.reshape(-1, jump)
prompt_max_logits_matrix = prompt_max_logits.reshape(-1, jump)

# get centered logits
logits_centered = logits - logits.mean(dim=-1, keepdim=True)
prompt_logits_centered = logits_centered[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits_centered = torch.max(logits_centered[0, :-1], dim=-1).values

prompt_logits_centered_matrix = prompt_logits_centered.reshape(-1, jump)
prompt_max_logits_centered_matrix = prompt_max_logits_centered.reshape(-1, jump)

# get probs matrix
probs = F.softmax(logits[0], dim=-1)  # shape [position, vocab]
prompt_probs = probs[range(prompt_dla.shape[0]), chosen_tokens_inds]
probs_matrix = prompt_probs.reshape(-1, jump)

for offset in range(jump):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,10))
    axs[0].plot(prompt_logits_matrix[:, offset].cpu().numpy())
    axs[0].plot(prompt_max_logits_matrix[:, offset].cpu().numpy(), 'r.')
    axs[0].grid()
    axs[0].set_title(f"Chosen and Maximal Logits")
    axs[0].set_xlabel("#Repetition")
    axs[0].set_ylabel("Logit")

    axs[1].plot(prompt_dla_matrix[:, offset].cpu().numpy(), label=["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(2) for head in range(8)])
    axs[1].grid()
    axs[1].set_title(f"DLA for chosen token")
    axs[1].legend()
    axs[1].set_xlabel("#Repetition")
    axs[1].set_ylabel("DLA")
    fig.suptitle(f"Offset {offset}, token: {model.to_string(chosen_tokens_inds[offset])}")

    # plt.figure(figsize=(10,8))
    # plt.plot((prompt_max_logits_matrix[:, offset] - prompt_logits_matrix[:, offset]).cpu().numpy())
    # plt.grid()
    # plt.title(f"Maximal to Chosen Logit Diff")
    # plt.xlabel("#Repetition")
    # plt.ylabel("Logit Diff")

    plt.figure(figsize=(10,8))
    plt.plot((probs_matrix[1:, offset] - probs_matrix[:-1, offset]).cumsum(dim=-1).cpu().numpy())
    plt.grid()
    plt.title(f"Probability Derivative Vs. #Repetition")
    plt.xlabel("#Repetition")
    plt.ylabel("Probability Diff")
    
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(probs_matrix.cpu().numpy(), label=[f"{model.to_string(token)}" for token in input_tokens[0,1:jump+1]])
plt.plot(0.5 * np.ones(shape=(probs_matrix.shape[0], 1)), 'k', label="$y=1/2$")
plt.grid()
plt.ylabel("Probability")
plt.xlabel("#Repetition")
plt.legend()
plt.show()

In [ ]:
logits_centered_matrix = logits_centered[0,:-1].reshape(-1, jump, logits_centered.shape[-1])
# final_resid = cache['blocks.1.hook_resid_post']
# logits_centered_matrix = final_resid[:-1].reshape(-1, jump, final_resid.shape[-1])

offset = 2
plt.figure(figsize=(10,8))
plt.plot(F.cosine_similarity(logits_centered_matrix[:-1,offset], logits_centered_matrix[1:,offset], dim=-1).cpu().numpy())
plt.grid()
plt.xlabel("#Repetition")
plt.ylabel("Cosine Similarity")
plt.title(f"Cosine Similarity Vs. Repetition, Offset {offset}")
plt.ylim(bottom=0.99, top=1.01)
plt.show()

In [ ]:
bla = dla[:,9,:]
probs_bla = F.softmax(bla, dim=-1)
plt.plot(probs_bla[0::2,1706].cpu().numpy())
plt.plot(probs_bla[0::2].max(dim=-1).values.cpu().numpy())
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
token_id = model.to_single_token(" met")
max_prob_ids = np.argmax(probs[0], axis=-1)
plt.plot(probs[0, :, token_id], "--.")
plt.plot(np.pad(max_prob_ids[1:], (1, 0)) == token_id, ".", label="Win?")
plt.grid()
plt.legend()

plt.show()

In [ ]:
# Get attention patterns and visualize them
layer = 0
print(type(cache))
attention_pattern = cache["attn", layer]
print(attention_pattern.shape) # [head_idx, destination, source]
str_tokens = model.to_str_tokens(prompt)

In [ ]:
print(f"Layer {layer} head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

In [ ]:
## Plot Attention Coeffs
head = 6
jump = 9
offset = 310
plt.plot(attention_pattern[head, offset].cpu().numpy(), '--o')
# plt.plot(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump][::-1], '--o')
plt.ylim(bottom=0)
plt.xlabel("Position")
plt.ylabel("Attention Weight")
plt.grid()
# plt.show()

# print(attention_pattern[head, -1].cpu().numpy().sum())

## Plot Token Relative Attention Coeff Vs. Repeats
relative_coeffs = [attention_pattern[head, pos].cpu().numpy()[pos-jump+1::-jump].sum() / attention_pattern[head, pos].cpu().numpy().sum() for pos in np.arange(jump+np.mod(offset, jump), len(attention_pattern[head, offset].cpu().numpy()), jump)]
# print(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump].sum() / attention_pattern[head, offset].cpu().numpy()[0])
plt.figure()
plt.plot(relative_coeffs)
plt.xlabel("#Repeats")
plt.ylabel("Relative Coeff")
plt.ylim(bottom=0, top=1)
plt.grid()
plt.show()

In [ ]:
layer = 1

# initial residual stream (embedding + positional)
x0 = cache["blocks.0.hook_resid_pre"] # shape: [batch, sequence_length, d_model]

# 1. The Residual Stream
# shape: [batch, sequence_length, d_model]
resid_pre = cache[f"blocks.{layer}.hook_resid_pre"] # Before attention
resid_post = cache[f"blocks.{layer}.hook_resid_post"] # After attention & addition

# 2. The Q, K, and V Projections
# shape: [batch, sequence_length, num_heads, d_head]
q_vectors = cache[f"blocks.{layer}.attn.hook_q"] # shape [position, head_idx, d_head]
k_vectors = cache[f"blocks.{layer}.attn.hook_k"]
v_vectors = cache[f"blocks.{layer}.attn.hook_v"]


# 3. Attention Scores
# hook_attn_scores: Unnormalized dot products (Q*K^T)
# hook_pattern: Normalized probabilities (after Softmax)
# shape: [batch, num_heads, query_pos, key_pos]
unnormalized_scores = cache[f"blocks.{layer}.attn.hook_attn_scores"]
attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]

# 4. Computed Outputs (Before being added to residual stream)
# hook_z: The output of the OV circuit before the final W_O projection
# hook_attn_out: The final output of each head after W_O projection (the exact H_t we modeled)
# shape (hook_attn_out): [batch, sequence_length, num_heads, d_model]
z_vectors = cache[f"blocks.{layer}.attn.hook_z"]
head_outputs = cache[f"blocks.{layer}.attn.hook_result"]

# 5. Final Logits
# Already returned from run_with_cache. shape: [batch, sequence_length, vocab_size]
print(f"Logits shape: {logits.shape}")

In [ ]:
## Plot Z norms Vs. Repeats

head = 6
jump = 5
z_norms = np.linalg.norm(z_vectors[:,head].cpu().numpy(), axis=-1)
z_norms = z_norms[1:] # remove the <BOS> token
# z_norms = z_norms[2:] # remove prefix

plt.figure(figsize=(10, 8))
plt.plot(np.reshape(z_norms, (-1, jump)))
plt.grid()
plt.xlabel("#Repeats")
plt.ylabel("Z norm")
plt.ylim(bottom=0)
plt.show()

In [ ]:
### Plot output norms vs. repeats
head = 6
jump = 8
o_norms = np.linalg.norm(head_outputs[:,head].cpu().numpy(), axis=-1)
o_norms = o_norms[1:] # remove the <BOS> token
# o_norms = o_norms[2:] # remove prefix

plt.figure(figsize=(10, 8))
plt.plot(np.reshape(o_norms, (-1, jump)))
plt.grid()
plt.xlabel("#Repeats")
plt.ylabel("Z norm")
plt.ylim(bottom=0)
plt.show()

In [ ]:
### Compute QK and OV matrices
layer = 0
head = 1
E = model.W_E.cpu() # shape: [vocab, d_model]
E_pos = model.W_pos.cpu() # shape: [n_context, d_model]
Q = model.W_Q[layer, head].cpu() # shape [d_model, d_head]
b_Q = model.b_Q[layer, head].cpu() # shape: [d_head]
K = model.W_K[layer, head].cpu() # shape [d_model, d_head]
b_K = model.b_K[layer, head].cpu() # shape: [d_head]
V = model.W_V[layer, head].cpu() # shape [d_model, d_head]
b_V = model.b_V[layer, head].cpu() # shape: [d_head]
O = model.W_O[layer, head].cpu() # shape [d_head, d_model]
b_O = model.b_O[layer].cpu() # shape: [d_model]
U = model.W_U.cpu() # shape [d_model, vocab]
b_U = model.b_U.cpu() # shape [vocab]

## Core matrices
OV_core = V @ O # shape [d_model, d_model]
QK_core = Q @ K.T # shaoe [d_model, d_model]

E_0 = E + E_pos[7]
E_1 = E + E_pos[8]

# Full matrices
QK_full = E @ QK_core @ E.T # shape [vocab, vocab]
QK_full_pos = E_pos @ QK_core @ E_pos.T # shape [n_context, n_context]
OV_full = E @ OV_core @ U
OV_full_pos = E_pos @ OV_core @ U


In [ ]:
# Softmax over the output dimension (dim=-1) to get the normalized distribution
# P_OV[i, j] = Probability of head predicting token j given source token i
temperature = 0.01
P_OV = F.softmax(OV_full / temperature, dim=-1) # Shape: [d_vocab, d_vocab]

E_virtual = P_OV @ E
K_virtual = E_virtual @ K

Q_standard = E @ Q

QK_virtual_full = Q_standard @ K_virtual.T # i (row): destination token - regular token embedding. j (column): source. OV weighted embeddings by token j. attention from token i to output of value of token j.

OV_standard = E @ V @ O
OV_virtual = E_virtual @ V @ O
# OV_virtual_diag = (OV_standard @ OV_virtual.T).diag()
OV_virtual_diag = F.cosine_similarity(OV_standard, OV_virtual, dim=-1)

feedback_scores = QK_virtual_full * QK_full * OV_virtual_diag.view(1, -1) # multiply each column by the F-OV score

In [ ]:
## Find Stable Tokens
d_vocab = model.cfg.d_vocab
layer = 0
heads = [3]
num_heads = len(heads)

ln1 = model.blocks[layer].ln1
ln_final = model.ln_final

E = model.W_E.cpu() # shape: [vocab, d_model]
Q = model.W_Q[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_Q = model.b_Q[layer, heads].cpu()
K = model.W_K[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_K = model.b_K[layer, heads].cpu()
V = model.W_V[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_V = model.b_V[layer, heads].cpu()
O = model.W_O[layer, heads].cpu() # shape: [num_heads, d_head, d_model]
b_O = model.b_O[layer].cpu()
U = model.W_U.cpu() # shape [d_model, vocab]
b_U = model.b_U.cpu() # shape [vocab]

# QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
# OV_core = V @ O  # shape [num_heads, d_model, d_model]

## Attention computation
attention_resid_pre = ln1(E) # shape: [L, d_model]

attention_weights = torch.eye(d_vocab).unsqueeze(0).repeat(num_heads, 1, 1) # shape: [num_heads, L, L], sums to 1 along dim=1

v = einops.einsum(attention_resid_pre, V, "L d_model, num_heads d_model d_head -> num_heads L d_head")
v = v + b_V.unsqueeze(1) # shape: [num_heads, L, d_head]

weighted_v = attention_weights.mT @ v # shape: [num_heads, L, d_head]
head_outputs = einops.einsum(weighted_v, O, "num_heads L d_head, num_heads d_head d_model -> num_heads L d_model")
attention_output = head_outputs.sum(dim=0) + b_O # shape: [L, d_model]

# value_bias = einops.einsum(model.b_V, model.W_O, "layer num_heads d_head, seq num_heads d_head d_model -> layer num_heads d_model")[:, heads].sum(dim=1).cpu() # shape: [1, d_model]
# attention_output = (attention_weights.mT @ attention_resid_pre @ OV_core).sum(dim=0) + b_O + value_bias # shape: [L, d_model]

# Unembedding
final_resid = attention_output + E # shape: [L, d_model]
final_resid_norm = ln_final(final_resid)
full_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [L, vocab]
max_OV_token_ind = full_logits_matrix.max(dim=-1).indices
token_is_stable = max_OV_token_ind == np.arange(model.cfg.d_vocab)
stable_tokens = token_is_stable.nonzero()

top_logits = full_logits_matrix.topk(2, dim=-1)
top_stable_logits_sorted = (top_logits.values[stable_tokens, 0] - top_logits.values[stable_tokens, 1]).squeeze().sort()

In [ ]:
sorted_index = 1000
token = stable_tokens[top_stable_logits_sorted.indices[sorted_index]]
value = top_stable_logits_sorted.values[sorted_index]
print(f"Token {token.item()}:{model.to_string(token)}, Top-2 Logit Diff: {value}")

plt.hist(top_stable_logits_sorted.values, bins=100)
plt.xlabel("Top-2 Logit Difference")
plt.ylabel("Counts")

plt.figure()
plt.plot(top_stable_logits_sorted.values)
plt.xlabel("Sorted Index")
plt.ylabel("Top-2 Logit Difference")
plt.show()


In [ ]:
layer = 0
heads = [3]
token1 = 10193
token2 = 2820
pi = torch.zeros(model.cfg.d_vocab, 1) # shape: [vocab, 1]
pi[token1] = pi[token2] = 0.5
# pi[token1] = 1

ln1 = model.blocks[layer].ln1
ln_final = model.ln_final

E = model.W_E.cpu() # shape: [vocab, d_model]
Q = model.W_Q[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_Q = model.b_Q[layer, heads].cpu()
K = model.W_K[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_K = model.b_K[layer, heads].cpu()
V = model.W_V[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_V = model.b_V[layer, heads].cpu()
O = model.W_O[layer, heads].cpu() # shape: [num_heads, d_head, d_model]
b_O = model.b_O[layer].cpu()
U = model.W_U.cpu() # shape [d_model, vocab]
b_U = model.b_U.cpu() # shape [vocab]

# QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
# OV_core = V @ O  # shape [num_heads, d_model, d_model]

## Attention computation
attention_resid_pre = ln1(E[[token1, token2]]) # shape: [L, d_model]

q = einops.einsum(attention_resid_pre, Q, "L d_model, num_heads d_model d_head -> num_heads L d_head")
q = q + b_Q.unsqueeze(1) # Broadcast bias over L: [num_heads, L, d_head]
k = einops.einsum(attention_resid_pre, K, "L d_model, num_heads d_model d_head -> num_heads L d_head")
k = k + b_K.unsqueeze(1) # Broadcast bias over L: [num_heads, L, d_head]

QK_raw = (q @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, L, L]
QK_weighted = pi[[token1, token2]] * torch.exp(QK_raw)
attention_weights = QK_weighted / QK_weighted.sum(dim=1, keepdim=True) # shape: [num_heads, L, L], sums to 1 along dim=1

v = einops.einsum(attention_resid_pre, V, "L d_model, num_heads d_model d_head -> num_heads L d_head")
v = v + b_V.unsqueeze(1) # shape: [num_heads, L, d_head]

weighted_v = attention_weights.mT @ v # shape: [num_heads, L, d_head]
head_outputs = einops.einsum(weighted_v, O, "num_heads L d_head, num_heads d_head d_model -> num_heads L d_model")
attention_output = head_outputs.sum(dim=0) + b_O # shape: [L, d_model]

# value_bias = einops.einsum(model.b_V, model.W_O, "layer num_heads d_head, seq num_heads d_head d_model -> layer num_heads d_model")[:, heads].sum(dim=1).cpu() # shape: [1, d_model]
# attention_output = (attention_weights.mT @ attention_resid_pre @ OV_core).sum(dim=0) + b_O + value_bias # shape: [L, d_model]

# Unembedding
final_resid = attention_output + E[[token1, token2]] # shape: [L, d_model]
final_resid_norm = ln_final(final_resid)
full_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [L, vocab]

pattern_stable = (full_logits_matrix[0].topk(1).indices == token2) and (full_logits_matrix[1].topk(1).indices == token1)
print("Pattern Stable!" if pattern_stable else "Pattern Unbstable")

In [ ]:
sorted_diag = full_logits_matrix.diag().sort()
stable_indices_sorted = [index for index in range(model.cfg.d_vocab) if sorted_diag.indices[index] in stable_tokens]
unstable_indices_sorted = [index for index in range(model.cfg.d_vocab) if sorted_diag.indices[index] not in stable_tokens]

plt.figure(figsize=(10,8))
plt.plot(stable_indices_sorted, sorted_diag.values[stable_indices_sorted], 'b.', markersize=0.1)
plt.plot(unstable_indices_sorted, sorted_diag.values[unstable_indices_sorted], 'r.', markersize=0.1)
# plt.plot([index in stable_tokens for index in full_logits_matrix.diag().sort().indices], 'r.')
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

matrix_to_plot = OV_full

tops_per_token = matrix_to_plot.topk(3, dim=-1)
strongest_tokens = tops_per_token.values[:,0].sort(descending=True)

y_ind = 0
y_offset = 200
x_ind = 0
x_offset = 200
v_max = matrix_to_plot[y_ind:y_ind+y_offset, x_ind:x_ind+x_offset].abs().max().item()
plt.imshow(matrix_to_plot[y_ind:y_ind+y_offset, x_ind:x_ind+x_offset], cmap="coolwarm", vmin=-v_max, vmax=v_max)
# plt.xticks(range(x_offset), range(x_ind, x_ind+x_offset))
# plt.yticks(range(y_offset), range(y_ind, y_ind+y_offset))
plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(F.softmax(matrix_to_plot, dim=-1).diag())
plt.grid()
plt.show()

In [ ]:
### Black Box Optimization
vocab_size = model.cfg.d_vocab
eps = 1e-10

# Turn off model requires grad
for p in model.parameters():
    p.requires_grad = False

# move to cpu
# model_cpu = model.cpu()

# Initialize optimization parameter
pi_init = torch.ones(vocab_size, device=device) * eps
# pi_init[432] = (0.9/(1-0.9)) * pi_init.sum()
# pi_init[6442] = 0.99
# pi_init[10193] = 0.47
# pi_init[1] = 0.01
pi_init = pi_from_context(model.to_tokens(patched_output)).clamp(min=1e-10)
pi_init = pi_init / pi_init.sum()
pi_logits = nn.Parameter(torch.log(pi_init)) # for Natural GD
# pi = nn.Parameter(pi_init) # for Exponentiated GD

# Define loss function
criterion = nn.MSELoss(reduction='sum')
# criterion = nn.L1Loss(reduction='sum')

heads = [0,1,2,3,4,5,6,7]

# Optimization parameters
lr = 1e0
n_iterations = 500
val_iterations = 250
print_iterations = 10
temperature = 1.0
chunk_size = 2**10
p = 0.9 # nucleus sampling parameter
# optimizer = torch.optim.Adam([pi_logits], lr=lr, eps=1e-15)
optimizer = torch.optim.SGD([pi_logits], lr=lr, momentum=0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max = n_iterations, eta_min=1e-3)

losses = []
full_losses = []
pi_list = []
pi_P_list = []
for i_iteration in tqdm(range(n_iterations)):
    # zero grad
    optimizer.zero_grad()

    pi = F.softmax(pi_logits, dim=-1)

    top_tokens = pi.topk(chunk_size).indices

    # forward pass
    # P_logits = pi_to_P_one_layer(pi, heads, model, chunk_size=512, temperature=temperature, p=p)
    # P = pi_to_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p) # Sparse matrix!

    # P = F.softmax(P_logits / temperature, dim=-1)

    # pi_col = pi.unsqueeze(1) # change to column vector [vocab, 1]
    # pi_P_col = torch.sparse.mm(P.t(), pi_col) # Perform Sparse @ Dense multiplication: P^T @ pi^T, shape: [vocab, 1]
    # pi_P = pi_P_col.squeeze(1)
    # pi_P = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
    pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens, temperature=temperature, p=p)
    pi_P = pi_P.squeeze()
    pi_P = pi_P.clamp(min=1e-15)

    # loss = criterion(pi, pi_P) * 1e3
    # loss = F.kl_div(pi.log(), pi_P.log(), reduction='batchmean', log_target=True)

    ## Forward KL
    # loss = 1e5 * F.kl_div(  # computes KL(p || pi_P)
    #     input=pi_P.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## Reverse KL
    # loss = 1e5 * F.kl_div(  # computes KL(pi_P || p)
    #     input=pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## JSD
    M_pi = 0.5 * (pi + pi_P)
    loss = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi.log(), 
        target=pi_P.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi.log(), 
        target=pi.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    if i_iteration % val_iterations == 0:
        with torch.no_grad():
            pi_P_full = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
            pi_P_full = pi_P_full.squeeze().clamp(min=1e-15)
            M_pi_full = 0.5 * (pi + pi_P_full)
            loss_full = 0.5 * 1e5 * (F.kl_div(  
                input=M_pi_full.log(), 
                target=pi_P_full.log(), 
                reduction='batchmean', 
                log_target=True
            ) + F.kl_div(  
                input=M_pi_full.log(), 
                target=pi.log(), 
                reduction='batchmean', 
                log_target=True
            )
            )

    losses.append(loss.item())
    full_losses.append(loss_full.item())
    pi_list.append(pi.detach().cpu())
    pi_P_list.append(pi_P.detach().cpu())
    loss.backward()

    # Natural Gradient scaling
    with torch.no_grad():
        g = pi_logits.grad
        # print(f"Original Gradient: {g.mean()}, {g.max()}")
        
        # Damping factor to prevent division by zero for dead tokens
        # (Because your distribution is sparse, this lambda is your lifeline)
        lmbda = 1e-10
        
        # 1. A_inv is just 1 / (p + lambda)
        A_inv = 1.0 / (pi + lmbda)
        
        # 2. Compute the three dot products/element-wise ops
        A_inv_g = A_inv * g                      # Vector: A^{-1} g
        A_inv_p = A_inv * pi                     # Vector: A^{-1} p
        
        p_A_inv_g = torch.sum(pi * A_inv_g)      # Scalar: p^T A^{-1} g
        p_A_inv_p = torch.sum(pi * A_inv_p)      # Scalar: p^T A^{-1} p
        
        # 3. Apply the Sherman-Morrison formula to get the exact preconditioned gradient
        numerator = A_inv_p * p_A_inv_g
        denominator = 1.0 - p_A_inv_p
        
        exact_natural_grad = A_inv_g + (numerator / denominator.clamp(min=1e-15))
        
        # 4. Overwrite the gradient
        pi_logits.grad = exact_natural_grad

        # print(f"A_inv: {A_inv.mean()}, {A_inv.max()}")
        # print(f"A_inv_p: {A_inv_p.mean()}, {A_inv_p.max()}")
        # print(f"p_A_inv_p: {p_A_inv_p}")
        # print(f"numerator: {numerator.mean()}, {numerator.max()}")
        # print(f"denominator: {denominator.mean()}, {denominator.max()}")
        # print(f"exact_natural_grad: {exact_natural_grad.mean()}, {exact_natural_grad.max()}")
        
        # 5. Clip to prevent wild swings from the lambda division
        torch.nn.utils.clip_grad_norm_([pi_logits], max_norm=1.0)

    # --- EXPONENTIATED GRADIENT UPDATE ---
    # with torch.no_grad():
    #     g = pi.grad
        
    #     # The Log-Space Trick for Bulletproof Numerical Stability
    #     # We add 'eps' to avoid log(0) for completely dead tokens
    #     z = torch.log(pi + 1e-12) - (lr * g)
        
    #     # Softmax naturally normalizes the vector so it sums to 1 again
    #     pi_new = F.softmax(z, dim=-1)
        
    #     # In-place copy to maintain the nn.Parameter reference for the next loop
    #     pi.copy_(pi_new)
        
    #     # Manually zero the gradient for the next iteration
    #     pi.grad.zero_()

    if i_iteration % print_iterations == 0:
        grad_mean = pi_logits.grad.abs().mean().item()
        grad_max = pi_logits.grad.abs().max().item()
        # grad_mean = pi.grad.abs().mean().item()
        # grad_max = pi.grad.abs().max().item()
        print(f"Gradient Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        # print(f"Loss: {loss.item():.2e}")
        print(f"Loss: {loss.item():.2e} || Full Loss: {loss_full.item():.2e}")
        print(f"pi:\t{pi.topk(5).values.tolist()},\t{pi.topk(5).indices.tolist()}")
        print(f"pi_P:\t{pi_P.topk(5).values.tolist()},\t{pi_P.topk(5).indices.tolist()}")

    optimizer.step()
    scheduler.step()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), losses)
plt.plot(np.arange(1, len(losses)+1), full_losses)
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), [pi_t[21462] for pi_t in pi_list])
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Pi[token]")
plt.title("Pi[token] Vs. Iteration")

plt.show()

In [ ]:
### Black Box Optimization - Exponentiated Gradient Descent
vocab_size = model.cfg.d_vocab
eps = 1e-10

# Turn off model requires grad
for p in model.parameters():
    p.requires_grad = False

# move to cpu
# model_cpu = model.cpu()

# Initialize optimization parameter
pi_init = torch.ones(vocab_size, device=device) * eps
# pi_init[432] = (0.9/(1-0.9)) * pi_init.sum()
pi_init[6442] = 0.99
# pi_init[10193] = 0.47
pi_init[1] = 0.01

pi_init = pi_from_context(model.to_tokens(patched_output)).clamp(min=1e-10)
pi_init = pi_init / pi_init.sum()
# pi_logits = nn.Parameter(torch.log(pi_init)) # for Natural GD
pi = nn.Parameter(pi_init) # for Exponentiated GD

# Define loss function
criterion = nn.MSELoss(reduction='sum')
# criterion = nn.L1Loss(reduction='sum')

heads = [0,1,2,3,4,5,6,7]

# Optimization parameters
lr = 3e-3
n_iterations = 1000
val_iterations = 250
print_iterations = 10
temperature = 1.0
chunk_size = 2**10
p = 0.9 # nucleus sampling parameter
# optimizer = torch.optim.Adam([pi_logits], lr=lr, eps=1e-15)
# optimizer = torch.optim.SGD([pi_logits], lr=lr, momentum=0)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max = n_iterations, eta_min=1e-1)

losses = []
losses_full = []
pi_list = []
pi_P_list = []
for i_iteration in tqdm(range(n_iterations)):
    # zero grad
    # optimizer.zero_grad()

    # pi = F.softmax(pi_logits, dim=-1)

    top_tokens = pi.topk(chunk_size).indices

    # forward pass
    # P_logits = pi_to_P_one_layer(pi, heads, model, chunk_size=512, temperature=temperature, p=p)
    # P = pi_to_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p) # Sparse matrix!

    # P = F.softmax(P_logits / temperature, dim=-1)

    # pi_col = pi.unsqueeze(1) # change to column vector [vocab, 1]
    # pi_P_col = torch.sparse.mm(P.t(), pi_col) # Perform Sparse @ Dense multiplication: P^T @ pi^T, shape: [vocab, 1]
    # pi_P = pi_P_col.squeeze(1)
    # pi_P = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
    pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens, temperature=temperature, p=p)
    pi_P = pi_P.squeeze()
    pi_P = pi_P.clamp(min=1e-15)

    # loss = criterion(pi, pi_P) * 1
    # loss = F.kl_div(pi.log(), pi_P.log(), reduction='batchmean', log_target=True)

    ## Forward KL
    # loss = 1e5 * F.kl_div(  # computes KL(p || pi_P)
    #     input=pi_P.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## Reverse KL
    # loss = 1e5 * F.kl_div(  # computes KL(pi_P || p)
    #     input=pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    # JSD
    M_pi = 0.5 * (pi + pi_P)
    loss = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi.log(), 
        target=pi_P.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi.log(), 
        target=pi.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    if i_iteration % val_iterations == 0:
        with torch.no_grad():
            pi_P_full = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
            pi_P_full = pi_P_full.squeeze().clamp(min=1e-15)
            M_pi_full = 0.5 * (pi + pi_P_full)
            loss_full = 0.5 * 1e5 * (F.kl_div(  
                input=M_pi_full.log(), 
                target=pi_P_full.log(), 
                reduction='batchmean', 
                log_target=True
            ) + F.kl_div(  
                input=M_pi_full.log(), 
                target=pi.log(), 
                reduction='batchmean', 
                log_target=True
            )
            )

    losses.append(loss.item())
    losses_full.append(loss_full.item())
    pi_list.append(pi.detach().cpu())
    pi_P_list.append(pi_P.detach().cpu())
    loss.backward()

    # Natural Gradient scaling
    # with torch.no_grad():
    #     g = pi_logits.grad
        
    #     # Damping factor to prevent division by zero for dead tokens
    #     # (Because your distribution is sparse, this lambda is your lifeline)
    #     lmbda = 1e-11
        
    #     # 1. A_inv is just 1 / (p + lambda)
    #     A_inv = 1.0 / (pi + lmbda)
        
    #     # 2. Compute the three dot products/element-wise ops
    #     A_inv_g = A_inv * g                      # Vector: A^{-1} g
    #     A_inv_p = A_inv * pi                     # Vector: A^{-1} p
        
    #     p_A_inv_g = torch.sum(pi * A_inv_g)      # Scalar: p^T A^{-1} g
    #     p_A_inv_p = torch.sum(pi * A_inv_p)      # Scalar: p^T A^{-1} p
        
    #     # 3. Apply the Sherman-Morrison formula to get the exact preconditioned gradient
    #     numerator = A_inv_p * p_A_inv_g
    #     denominator = 1.0 - p_A_inv_p
        
    #     exact_natural_grad = A_inv_g + (numerator / denominator)
        
    #     # 4. Overwrite the gradient
    #     pi_logits.grad = exact_natural_grad
        
    #     # 5. Clip to prevent wild swings from the lambda division
    #     torch.nn.utils.clip_grad_norm_([pi_logits], max_norm=1.0)
    if i_iteration % print_iterations == 0:
        grad_mean = pi.grad.abs().mean().item()
        grad_max = pi.grad.abs().max().item()
        print(f"Gradient Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        print(f"Loss: {loss.item():.2e}")
        print(f"pi top-5: {pi.topk(5)}")
        print(f"pi_P top-5: {pi_P.topk(5)}")

    # --- EXPONENTIATED GRADIENT UPDATE ---
    with torch.no_grad():
        g = pi.grad
        
        # The Log-Space Trick for Bulletproof Numerical Stability
        # We add 'eps' to avoid log(0) for completely dead tokens
        z = torch.log(pi + 1e-12) - (lr * g)
        
        # Softmax naturally normalizes the vector so it sums to 1 again
        pi_new = F.softmax(z, dim=-1)
        
        # In-place copy to maintain the nn.Parameter reference for the next loop
        pi.copy_(pi_new)
        
        # Manually zero the gradient for the next iteration
        pi.grad.zero_()

    # grad_mean = pi_logits.grad.abs().mean().item()
    # grad_max = pi_logits.grad.abs().max().item()

    # optimizer.step()
    # scheduler.step()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), losses)
# plt.plot(np.arange(1, len(losses)+1), losses_full)
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), [pi_t[21462] for pi_t in pi_list])
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Pi[token]")
plt.title("Pi[token] Vs. Iteration")

plt.show()

In [ ]:
plt.hist(pi_list[-1].topk(100).values.log(), bins=50)
plt.hist(pi_P_list[-1].topk(100).values.log(), bins=50)
plt.show()

### Examining Trajectories

In [ ]:
## compute loss at positional and non-positional fixed points
heads = [0,1,2,3,4,5,6,7]
with torch.no_grad():

    pi_positional = pi_from_context(model.to_tokens(clean_output)).clamp(min=1e-15)
    # pi_positional = torch.zeros(model.cfg.d_vocab, device=device).clamp(min=1e-15)
    # pi_positional[2320] = 1
    pi_P_positional = pi_to_pi_P_one_layer(pi_positional, heads, model, chunk_size=2048, temperature=1, p=0.9).squeeze().clamp(min=1e-15)
    M_pi_pos = 0.5 * (pi_positional + pi_P_positional)
    loss_pos = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi_pos.log(), 
        target=pi_P_positional.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi_pos.log(), 
        target=pi_positional.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    pi_nonpositional = pi_from_context(model.to_tokens(patched_output)).clamp(min=1e-15)
    pi_P_nonpositional = pi_to_pi_P_one_layer(pi_nonpositional, heads, model, chunk_size=2048, temperature=1, p=0.9).squeeze().clamp(min=1e-15)
    M_pi_nonpos = 0.5 * (pi_nonpositional + pi_P_nonpositional)
    loss_nonpos = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi_nonpos.log(), 
        target=pi_P_nonpositional.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi_nonpos.log(), 
        target=pi_nonpositional.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

print(f"Positional Pi Loss:\t{loss_pos}")
print(f"Non-Positional Pi Loss:\t{loss_nonpos}")

In [ ]:
with torch.no_grad():
    pi_t_clean = pi_t_from_context(model.to_tokens(clean_output)[0])
    pi_t_patched = pi_t_from_context(model.to_tokens(patched_output)[0])
plt.figure(figsize=(10,8))
plt.plot(pi_t_clean[:, 4171].cpu().numpy())
plt.plot(pi_t_patched[:, 4171].cpu().numpy())
plt.grid()
plt.show()

### PCA Vizualizations

In [ ]:
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

pca = PCA(n_components=17)

# pca.fit(last_token_internal_activations.cpu().detach().numpy())

embeddings = pca.fit_transform(prompt_dla.cpu().numpy())
# embeddings = pca.fit_transform(last_token_internal_activations.cpu().detach().numpy())
# embeddings = pca.transform(last_token_internal_activations.cpu().detach().numpy())
# embeddings = pca.transform(last_token_activations.cpu().detach().numpy())
# embeddings2 = pca.transform(last_token_activations2.cpu().detach().numpy())
# mu_p_pca = pca.transform(last_token_activations.cpu().numpy())

plt.figure(figsize=(8, 5))
plt.scatter(embeddings[:, 0], embeddings[:, 1], marker='o')
# plt.scatter(embeddings[:, 0], embeddings[:, 1], marker='o', label='Mask 2', s=5)
# plt.scatter(embeddings[M1_mask, 0], embeddings[M1_mask, 1], marker='o', label='Mask 1')
# plt.scatter(embeddings[M2_mask, 0], embeddings[M2_mask, 1], marker='o', label='Mask 2')
# plt.scatter(embeddings2[:, 0], embeddings2[:, 1], marker='o', label='Encoder Mu (GT)')
# plt.scatter(mu_p_pca[:, 0], mu_p_pca[:, 1], marker='o', label='Transition Mu')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'PCA of Mu following {NT_id2token[previous_token_id]}, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.title(f'PCA of DLA Embeddings, Explained Variance: {pca.explained_variance_ratio_[:2].sum():.3f}')
plt.xlabel('PC1')
plt.ylabel('PC2')
# plt.xticks(steps)
plt.grid(True)
# plt.legend()
plt.axis("equal")

# fig = plt.figure(figsize=(15, 12))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(
#     embeddings[combined_mask_1, 0],
#     embeddings[combined_mask_1, 1],
#     embeddings[combined_mask_1, 2],
#     label="Mask 1",
#     s=5
# )
# ax.scatter(
#     embeddings[combined_mask_2, 0],
#     embeddings[combined_mask_2, 1],
#     embeddings[combined_mask_2, 2],
#     label="Mask 2",
#     s=5
# )
# ax.scatter(
#     embeddings[M1_mask, 0],
#     embeddings[M1_mask, 1],
#     embeddings[M1_mask, 2],
#     label="Mask 1"
# )
# ax.scatter(
#     embeddings[M2_mask, 0],
#     embeddings[M2_mask, 1],
#     embeddings[M2_mask, 2],
#     label="Mask 2"
# )

# ax.set_xlabel("PC1")
# ax.set_ylabel("PC2")
# ax.set_zlabel("PC3")
# ax.set_title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_[:3].sum():.3f}')
# ax.legend()

plt.show()

In [ ]:
## Plot V-vectors and Z-vectors Cosine Similarity Vs. Position

v_cosine_sim = F.cosine_similarity(v_vectors[-2, head], z_vectors[:,head], dim=-1)
plt.figure(figsize=(10, 8))
plt.plot(v_cosine_sim.cpu().numpy())
# plt.plot(z_norms[offset::jump])
plt.grid()
plt.xlabel("Position")
plt.ylabel("Cosine Sim")
plt.ylim(bottom=0, top=1)
plt.show()

In [ ]:
## Causal Patching Experiments

# Setup our experiment variables
clean_prompt =   " I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M"
clean_answer = " Q" # Notice the leading space! Tokenization matters.

corrupt_prompt = " I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M"
corrupt_answer = " A"

target_layer = 1
target_head = 7 # Let's say we suspect L1H4 is an induction head
target_position = -1 # The position of the second name token

# --- Experiment 1: Wide Patching (All positions) ---
print("\n--- Running Wide Patching ---")
results_wide = run_attention_patching(
    model, clean_prompt, corrupt_prompt, clean_answer, corrupt_answer,
    layer=target_layer, head_idx=target_head, position=None
)
print(f"Clean Logit Diff:   {results_wide['clean_diff']:.4f}")
print(f"Corrupt Logit Diff: {results_wide['corrupt_diff']:.4f}")
print(f"Patched Logit Diff: {results_wide['patched_diff']:.4f}")
print(f"Recovery Score:     {results_wide['recovery_score']:.2%}")

# # --- Experiment 2: Position-Specific Patching ---
# print(f"\n--- Running Position-Specific Patching (Pos: {target_position}) ---")
# results_pos = run_attention_patching(
#     model, clean_prompt, corrupt_prompt, clean_answer, corrupt_answer,
#     layer=target_layer, head_idx=target_head, position=target_position
# )
# print(f"Patched Logit Diff: {results_pos['patched_diff']:.4f}")
# print(f"Recovery Score:     {results_pos['recovery_score']:.2%}")

In [ ]:
from tqdm.auto import tqdm

def run_scaling_patching_experiment(
    model: HookedTransformer, 
    base_seq: str, 
    clean_target: str, 
    corrupt_target: str, 
    max_repetitions: int = 10,
    prefix: str|None = None
):
    """
    Runs causal tracing across all layers and heads for increasing repetitions.
    Returns a numpy array of shape [n_layers, n_heads, max_repetitions]
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    
    # Initialize the results array. 
    # Shape: [layer, head_idx, n_repetitions]
    # Note: Index 0 on the last dimension corresponds to N=1 repetition.
    recovery_scores = np.zeros((n_layers, n_heads, max_repetitions))
    
    clean_answer_id = model.to_single_token(clean_target)
    corrupt_answer_id = model.to_single_token(corrupt_target)

    min_repetitions = 0

    # Loop over the number of repetitions
    for n_rep in tqdm(range(min_repetitions, max_repetitions + 1), desc="Repetitions"):
        
        # 1. Dynamically construct the prompts for this N
        if prefix is not None:
            clean_prompt = prefix + (base_seq + clean_target) * n_rep + base_seq
            corrupt_prompt = prefix + (base_seq + corrupt_target) * n_rep + base_seq
        else:
            clean_prompt = (base_seq + clean_target) * n_rep + base_seq
            corrupt_prompt = (base_seq + corrupt_target) * n_rep + base_seq
        
        clean_tokens = model.to_tokens(clean_prompt)
        corrupt_tokens = model.to_tokens(corrupt_prompt)
        
        # 2. Run clean and corrupt baseline ONCE per N to save compute
        clean_logits, clean_cache = model.run_with_cache(clean_tokens)
        corrupt_logits = model(corrupt_tokens)
            
        clean_diff = get_logit_diff(clean_logits, clean_answer_id, corrupt_answer_id)
        corrupt_diff = get_logit_diff(corrupt_logits, clean_answer_id, corrupt_answer_id)
        
        # 3. Iterate over all layers and heads
        for layer in range(n_layers):
            hook_name = f"blocks.{layer}.attn.hook_z"

            for head_idx in range(n_heads):    
                
                # We use wide patching (position=None) across the whole sequence
                hook_fn = partial(
                    patch_head_hook, 
                    clean_cache=clean_cache, 
                    head_idx=head_idx, 
                    position=None 
                )
                
                patched_logits = model.run_with_hooks(
                    corrupt_tokens,
                    fwd_hooks=[(hook_name, hook_fn)]
                )
                
                patched_diff = get_logit_diff(patched_logits, clean_answer_id, corrupt_answer_id)
                
                # Calculate and store the recovery score
                # Handle edge case where clean_diff == corrupt_diff (e.g. at N=0 if model knows nothing)
                if clean_diff == corrupt_diff:
                    score = 0.0
                else:
                    score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)
                    
                recovery_scores[layer, head_idx, n_rep] = score

    return recovery_scores

base_sequence = " seems right. It"
clean_target = " just"
corrupt_target = " like"
prefix = "I always thought that I would make it to the moon. Why wouldn't I? It just"

max_reps = 100 #

print("Running experiment...")
results_matrix = run_scaling_patching_experiment( # shape [layer, head, n_repetition]
    model, 
    base_seq=base_sequence, 
    clean_target=clean_target, 
    corrupt_target=corrupt_target,
    max_repetitions=max_reps,
    prefix=prefix
)

print(f"Finished! Matrix shape: {results_matrix.shape}")

In [ ]:
# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(results_matrix[layer][i_head], label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Recovery Score [%]")
    plt.title("Recovery Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

In [ ]:
layer = 1
plt.figure(figsize=(10, 8))

plt.plot(results_matrix[layer][6] + results_matrix[layer][7], label="L1H6 + L1H7")

plt.xlabel("# Repetitions")
plt.ylabel("Recovery Score [%]")
plt.title("Recovery Score vs. # Repetitions")
plt.legend()
plt.grid()

In [ ]:
def run_natural_sufficiency_experiment(
    model: HookedTransformer, 
    base_seq: str, 
    clean_target: str, 
    long_natural_text: str,
    max_repetitions: int = 10,
    prefix: str = "",
    position=None
):
    """
    Runs causal tracing using a natural text corrupt baseline.
    Returns two arrays of shape [n_layers, n_heads, max_repetitions]:
      1. Logit Difference Recovery Scores
      2. Probability Recovery Scores
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    
    with torch.no_grad():
        # Initialize results matrices
        recovery_scores_diff = np.zeros((n_layers, n_heads, max_repetitions+1))
        recovery_scores_prob = np.zeros((n_layers, n_heads, max_repetitions+1))
        clean_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        corrupt_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        patched_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        patched_winner = np.zeros((n_layers, n_heads, max_repetitions+1))
        
        clean_answer_id = model.to_single_token(clean_target)
        
        # Tokenize the natural text ONCE outside the loop
        # We will slice this tensor to match the clean prompt's length at each N
        natural_tokens = model.to_tokens(long_natural_text)
        
        for n_rep in tqdm(range(0, max_repetitions + 1), desc="Repetitions"):
            
            # 1. Construct the Clean Prompt and tokenize
            clean_prompt = prefix + (base_seq + clean_target) * n_rep + base_seq
            clean_tokens = model.to_tokens(clean_prompt)
            seq_len = clean_tokens.shape[1]
            
            # 2. Slice the Natural Text for the Corrupt Prompt
            # Ensure we have enough natural text to cover the prompt AND the next token
            if natural_tokens.shape[1] <= seq_len:
                raise ValueError(
                    f"long_natural_text is too short! Need at least {seq_len + 1} tokens "
                    f"for N={n_rep}, but only have {natural_tokens.shape[1]}."
                )
                
            corrupt_tokens = natural_tokens[:, :seq_len]
            
            # The corrupt target is whatever token naturally follows the slice
            corrupt_answer_id = natural_tokens[0, seq_len].item()
            
            # 3. Baseline Forward Passes
            clean_logits, clean_cache = model.run_with_cache(clean_tokens)
            corrupt_logits = model(corrupt_tokens)
            
            # 4. Calculate Baseline Metrics
            # --- Metric 1: Logit Difference ---
            clean_diff = get_logit_diff(clean_logits, clean_answer_id, corrupt_answer_id)
            corrupt_diff = get_logit_diff(corrupt_logits, clean_answer_id, corrupt_answer_id)
            
            # --- Metric 2: Probability ---
            clean_prob = get_probability(clean_logits, clean_answer_id)
            corrupt_prob = get_probability(corrupt_logits, clean_answer_id)
            
            # 5. Patching Loop
            for layer in range(n_layers):
                for head_idx in range(n_heads):
                    hook_name = f"blocks.{layer}.attn.hook_z"
                    
                    hook_fn = partial(
                        patch_head_hook, 
                        clean_cache=clean_cache, 
                        head_idx=head_idx, 
                        position=position # Wide patching
                    )
                    
                    patched_logits = model.run_with_hooks(
                        corrupt_tokens,
                        fwd_hooks=[(hook_name, hook_fn)]
                    )
                    
                    # --- Evaluate Metric 1 (Diff) ---
                    patched_diff = get_logit_diff(patched_logits, clean_answer_id, corrupt_answer_id)
                    if clean_diff == corrupt_diff:
                        diff_score = 0.0
                    else:
                        diff_score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)
                    recovery_scores_diff[layer, head_idx, n_rep] = diff_score
                    
                    # --- Evaluate Metric 2 (Prob) ---
                    patched_prob = get_probability(patched_logits, clean_answer_id)
                    # if layer == 1 and head_idx == 6:
                    #     print(f"L1H6 repetition {n_rep}: clean prob: {clean_prob}, corrupt prob: {corrupt_prob}, patched prob: {[patched_prob]}")
                    if clean_prob == corrupt_prob:
                        prob_score = 0.0
                    else:
                        prob_score = (patched_prob - corrupt_prob) / (clean_prob - corrupt_prob)
                    recovery_scores_prob[layer, head_idx, n_rep] = prob_score
                    clean_probs[layer, head_idx, n_rep] = clean_prob
                    corrupt_probs[layer, head_idx, n_rep] = corrupt_prob
                    patched_probs[layer, head_idx, n_rep] = patched_prob
                    
                    full_patched_probs = F.softmax(patched_logits[0, -1, :], dim=-1)
                    patched_winner[layer, head_idx, n_rep] = patched_prob == np.max(full_patched_probs.cpu().numpy())

    return recovery_scores_diff, recovery_scores_prob, clean_probs, corrupt_probs, patched_probs, patched_winner


# Define experiment parameters
prefix = "Hello there, nice to meet you. I am a nice and"
base_sequence = " helpful robot. I am a nice"
clean_target = " and"
max_reps = 120
position = -1

# Needs to be quite long to cover max_reps of the sequence!
# E.g., for N=8, len = 8 * 5 tokens + 4 tokens = 44 tokens. 
# Let's provide a Wikipedia snippet as our natural baseline.
wiki_text = (
    """On 11 April 1951, U.S. president Harry S. Truman relieved General of the Army Douglas MacArthur of his commands after MacArthur made public statements that contradicted the administration's policies. MacArthur was a popular hero of World War II who was then commander of United Nations Command forces fighting in the Korean War, and his relief remains a controversial topic in the field of civil-military relations. United States General of the Army Douglas MacArthur shakes hands with US president Harry Truman at the Wake Island Conference, seven months before Truman relieved MacArthur from command. MacArthur led the Allied forces in the Southwest Pacific during World War II, and after the war was in charge of the occupation of Japan. In the latter role, MacArthur was able to accumulate considerable power over the civil administration of Japan. Eventually, he gained a level of political experience that was unprecedented and yet to be repeated by anyone else actively serving as a flag officer in the U.S. military. After North Korea invaded South Korea in June 1950, starting the Korean War, MacArthur was designated commander of the United Nations forces defending South Korea. He conceived and executed the amphibious assault at Inchon on 15 September 1950, but when he followed up his victory with a full-scale invasion of North Korea, China inflicted a series of defeats, compelling him to withdraw from North Korea. By April 1951, the military situation had stabilized, but MacArthur publicly criticized the administration's policies, leading Truman to have MacArthur relieved of his command. An apolitical military is an American tradition. The principle of civilian control of the military was also ingrained. Civilian control was an issue considering the constitutional division of powers between the president as commander-in-chief, and Congress with its power to raise armies, maintain a navy, and declare war. This was also an era when the rising complexity of military technology led to the creation of a professional military, and American forces were employed overseas in large numbers. The Armed Services Committee and the Foreign Relations Committee of the U.S. Senate held a joint inquiry into the military situation and the circumstances surrounding MacArthur's relief, and concluded that "the removal of General MacArthur was within the constitutional powers of the President but the circumstances were a shock to national pride".[1] In having MacArthur relieved for failing to "respect the authority of the President" by privately communicating with Congress, Truman upheld the president's role as preeminent. Harry S. Truman, then the vice president of the United States, became president of the United States on the death of Franklin D. Roosevelt in 1945, and won an unexpected victory to a full term in the 1948 presidential election. He was the only president who served after 1897 without a college degree. Although not highly educated, Truman was well read. When his high school friends went off to the state university in 1901, he enrolled in a local business school, but only lasted a semester. He later took night courses at the Kansas City Law School, but dropped out. Truman attempted to gain admission to the United States Military Academy at West Point, but was rejected for his poor eyesight. He was proud of his military service in the artillery during World War I, and continued to hold a reserve commission, eventually achieving the rank of colonel. Instead of professional soldiers, Truman selected two National Guardsmen, Harry H. Vaughan and Louis H. Renfrow, as his military aides. Truman once remarked that he did not understand how the US Army could "produce men such as Robert E. Lee, John J. Pershing, Eisenhower, and Bradley and at the same time produce Custers, Pattons, and MacArthur". During the 1949 Revolt of the Admirals, several naval officers publicly disagreed with the administration's policy over cuts to naval aviation and amphibious warfare capability, resulting in the relief of the Chief of Naval Operations, Admiral Louis Denfeld, and his replacement by Admiral Forrest Sherman. In testimony before the House Armed Services Committee investigation into the affair in October 1949, the chairman of the Joint Chiefs of Staff, General Omar Bradley, doubted that there would ever be another large-scale amphibious operation. In stature and seniority, General of the Army Douglas MacArthur was the Army's foremost general. The son of Lieutenant General Arthur MacArthur Jr., a recipient of the Medal of Honor for action during the American Civil War, he had graduated at the top of his West Point class of 1903, but never attended an advanced service school except for the engineer course in 1908. He had a distinguished combat record in World War I, and had served as Chief of Staff of the United States Army from 1930 to 1935, working closely with Presidents Herbert Hoover and Franklin Roosevelt, despite occasional clashes over the military budget. He would later compare Roosevelt's "extraordinary self-control" with Truman's "violent temper and paroxysms of ungovernable rage". Apart from his World War I-era service in Mexico and Europe, his overseas postings had been in Asia and the Pacific. During World War II, he had become a national hero and had been awarded the Medal of Honor for the unsuccessful defense of the Philippines in the Battle of Bataan. He had commanded the Allied armies in the New Guinea Campaign and Philippines Campaign, fulfilling his famous promise to return to the Philippines. In 1944 and 1948, he had been considered a possible Republican candidate for president. After the war, as the Supreme Commander of the Allied Powers, he had overseen the occupation of Japan and played an important part in the post-war political and social transformation of that country. By 1950, the occupation of Japan was winding down, but MacArthur remained in the country as Commander-in-Chief Far East, a post to which he had been appointed by Truman in 1945. MacArthur had to deal with deep cuts in the defense budget that saw his troop numbers decline from 300,000 in 1947 to 142,000 in 1948. Despite his protests, further reductions followed and, by June 1950, there were only 108,000 troops in his Far East Command. Cuts in funds and personnel produced shortages of serviceable equipment. Of the Far East Command's 18,000 jeeps, 10,000 were unserviceable; of its 13,780 2+1/2-ton 6x6 trucks, only 4,441 were serviceable. On the positive side, Far East Command initiated a program of reclaiming and refurbishing war materiel from abandoned stocks throughout the Pacific. This had not only recovered a great deal of valuable stores and equipment, it had also generated a useful repair and rebuilding industry in Japan. Meanwhile, the shift away from occupation duties had permitted a greater focus on training for combat. North Korea invaded South Korea on 25 June 1950, starting the Korean War. In response to an urgent request from the Korean Military Advisory Group for more ammunition, MacArthur, on his own initiative, ordered the transport ship MSTS Sgt. George D Keathley, then in harbor in Yokohama, to be loaded with ammunition and to sail for Pusan. President Truman met with the Joint Chiefs of Staff and other advisors that day at Blair House, his temporary residence during the White House Reconstruction, and approved the actions already taken by MacArthur and Secretary of State Dean Acheson. At another meeting at Blair House held on the evening of 26 June, amid reports of a rapidly deteriorating situation in South Korea, Truman approved the use of air and naval forces against military targets south of the 38th parallel north. Subsequently, on 27 June, the United Nations Security Council passed Resolution 83, which recommended that "members of the United Nations furnish such assistance to the Republic of Korea as may be necessary to repel the armed attack and to restore international peace and security in the area". The South Korean capital of Seoul fell on 28 June. The next day, Truman authorized air and naval operations north of the 38th parallel, which MacArthur had already ordered. However it was not until 30 June, following a sobering report on the military situation from MacArthur, that Truman finally authorized the use of ground forces. On 8 July, on the advice of the Joint Chiefs of Staff, Truman appointed MacArthur commander of United Nations Command in South Korea. He remained Commander-in-Chief Far East and Supreme Commander of the Allied Powers. MacArthur was forced to commit his forces in Japan to what he later described as a "desperate rearguard action". In July, Truman sent the Chief of Staff of the Army, General J. Lawton Collins, and the Chief of Staff of the Air Force, General Hoyt S. Vandenberg, to report on the situation. They met with MacArthur and his chief of staff, Major General Edward Almond, in Tokyo on 13 July. MacArthur impressed on them the danger of underestimating the North Koreans, whom he characterized as "well-equipped, well-led, and battle-trained, and which have at times out-numbered our troops by as much as twenty to one". He proposed to first halt the North Korean advance and then counterattack, enveloping the North Koreans with an amphibious operation, but the timing was dependent on the arrival of reinforcements from the United States. Bradley raised the possibility of using nuclear weapons in Korea at a Joint Chiefs of Staff meeting on 9 July 1950 at Eisenhower's instigation, but there was no support for the idea. The Army staff sent a cable to Collins in Tokyo suggesting that he seek out MacArthur's opinion. In a teleconference on 13 July, Major General Charles L. Bolte proposed sending nuclear weapons. MacArthur had already turned down Air Force proposals to firebomb North Korean cities, and suggested that atomic bombs could be used to isolate North Korea by taking out bridges and tunnels. The Army staff considered this impractical. On 28 July, the Joint Chiefs decided to send ten nuclear-capable B-29 bombers of the 9th Bombardment Wing to Guam as a deterrent to Chinese action against Taiwan. Truman publicly denied that he was considering the use of nuclear weapons in Korea, but authorized the transfer to Guam of atomic bombs without their fissile cores. The deployment did not go well; one of the bombers crashed on takeoff from Fairfield-Suisun Air Force Base in California on 5 August, killing the mission commander, Brigadier General Robert F. Travis, and 18 others. The remaining nine bombers remained in Guam until 13 September, when they returned to the United States. The bomb assemblies stayed behind. At a press conference on 13 July, Truman was asked if United States forces would cross the 38th parallel into North Korea, and he replied that he would "make that decision when it becomes necessary to do it". Some of his advisors, including the Assistant Secretary of State for Far Eastern Affairs, Dean Rusk, and the director of the Office of Northeast Asian Affairs, John M. Allison, argued that Security Council Resolution 83 provided a legal basis for the invasion of North Korea. Others, such as George F. Kennan and Paul Nitze, disagreed. Along with the legality, the administration also had to consider the danger of intervention by the Soviet Union or the People's Republic of China if United Nations forces approached their borders."""
)

print("Running natural text patching experiment...")
diff_matrix, prob_matrix, clean_probs_matrix, corrupt_probs_matrix, patched_probs_matrix, patched_winner_matrix = run_natural_sufficiency_experiment(
    model, 
    base_seq=base_sequence, 
    clean_target=clean_target, 
    long_natural_text=wiki_text,
    max_repetitions=max_reps,
    prefix=prefix,
    position=position
)

# target_layer = 1
# target_head = 6

# scores_diff = diff_matrix[target_layer, target_head, :]
# scores_prob = prob_matrix[target_layer, target_head, :]

# print(f"{'N':<5} | {'Diff Recovery':<15} | {'Prob Recovery':<15}")
# print("-" * 40)
# for n in range(max_reps):
#     print(f"{n+1:<5} | {scores_diff[n]:7.2%}         | {scores_prob[n]:7.2%}")

In [ ]:
# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(diff_matrix[layer][i_head], "--.", label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Logits Recovery Score")
    plt.title("Logits Recovery Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(prob_matrix[layer][i_head], "--.", label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Probability Recovery Score")
    plt.title("Probability Recovery Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

In [ ]:
from scipy.ndimage import uniform_filter1d as movmean

w = 7
plt.plot(movmean(diff_matrix[1][6], size=w, mode="nearest"))
plt.show()

In [ ]:
layer = 1
i_head = 6

plt.figure(figsize=(10,8))
plt.plot(clean_probs_matrix[layer][i_head], label=f"Clean Probs")
plt.plot(corrupt_probs_matrix[layer][i_head], label=f"Corrupt Probs")
plt.plot(patched_probs_matrix[layer][i_head], label=f"Patched Probs")
plt.plot(patched_winner_matrix[layer][i_head], "r.", label=f"Patched Won?")

plt.xlabel("# Repetitions")
# plt.ylabel("Logits Recovery Score [%]")
# plt.title("Logits Recovery Score vs. # Repetitions")
plt.legend()
plt.grid()

In [ ]:
def run_natural_neccesity_experiment(
    model: HookedTransformer, 
    base_seq: str, 
    clean_target: str, 
    long_natural_text: str,
    max_repetitions: int = 10,
    prefix: str = "",
    position=None,
    knockout=False
):
    """
    Runs causal tracing using a natural text corrupt baseline.
    Returns two arrays of shape [n_layers, n_heads, max_repetitions]:
      1. Logit Difference Recovery Scores
      2. Probability Recovery Scores
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    
    with torch.no_grad():
        # Initialize results matrices
        destruction_scores_diff = np.zeros((n_layers, n_heads, max_repetitions+1))
        destruction_scores_prob = np.zeros((n_layers, n_heads, max_repetitions+1))
        clean_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        corrupt_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        patched_probs = np.zeros((n_layers, n_heads, max_repetitions+1))
        patched_winner = np.zeros((n_layers, n_heads, max_repetitions+1))
        
        clean_answer_id = model.to_single_token(clean_target)
        
        # Tokenize the natural text ONCE outside the loop
        # We will slice this tensor to match the clean prompt's length at each N
        natural_tokens = model.to_tokens(long_natural_text)
        
        for n_rep in tqdm(range(0, max_repetitions + 1), desc="Repetitions"):
            
            # 1. Construct the Clean Prompt and tokenize
            clean_prompt = prefix + (base_seq + clean_target) * n_rep + base_seq
            clean_tokens = model.to_tokens(clean_prompt)
            seq_len = clean_tokens.shape[1]
            
            # 2. Slice the Natural Text for the Corrupt Prompt
            # Ensure we have enough natural text to cover the prompt AND the next token
            if natural_tokens.shape[1] <= seq_len:
                raise ValueError(
                    f"long_natural_text is too short! Need at least {seq_len + 1} tokens "
                    f"for N={n_rep}, but only have {natural_tokens.shape[1]}."
                )
                
            corrupt_tokens = natural_tokens[:, :seq_len]
            
            # The corrupt target is whatever token naturally follows the slice
            corrupt_answer_id = natural_tokens[0, seq_len].item()
            
            # 3. Baseline Forward Passes
            corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_tokens)
            clean_logits = model(clean_tokens)
            
            # 4. Calculate Baseline Metrics
            # --- Metric 1: Logit Difference ---
            clean_diff = get_logit_diff(clean_logits, clean_answer_id, corrupt_answer_id)
            corrupt_diff = get_logit_diff(corrupt_logits, clean_answer_id, corrupt_answer_id)
            
            # --- Metric 2: Probability ---
            clean_prob = get_probability(clean_logits, clean_answer_id)
            corrupt_prob = get_probability(corrupt_logits, clean_answer_id)
            
            # 5. Patching Loop
            for layer in range(n_layers):
                for head_idx in range(n_heads):
                    hook_name = f"blocks.{layer}.attn.hook_z"
                    
                    hook_fn = partial(
                        patch_head_hook, 
                        clean_cache=corrupt_cache, 
                        head_idx=head_idx, 
                        position=position,
                        knockout=knockout
                    )
                    
                    patched_logits = model.run_with_hooks(
                        clean_tokens,
                        fwd_hooks=[(hook_name, hook_fn)]
                    )
                    
                    # --- Evaluate Metric 1 (Diff) ---
                    patched_diff = get_logit_diff(patched_logits, clean_answer_id, corrupt_answer_id)
                    if clean_diff == corrupt_diff:
                        diff_score = 0.0
                    else:
                        diff_score = (clean_diff - patched_diff) / (clean_diff - corrupt_diff) # "destruction" / neccesity score
                    destruction_scores_diff[layer, head_idx, n_rep] = diff_score
                    
                    # --- Evaluate Metric 2 (Prob) ---
                    patched_prob = get_probability(patched_logits, clean_answer_id)
                    # if layer == 1 and head_idx == 6:
                    #     print(f"L1H6 repetition {n_rep}: clean prob: {clean_prob}, corrupt prob: {corrupt_prob}, patched prob: {[patched_prob]}")
                    if clean_prob == corrupt_prob:
                        prob_score = 0.0
                    else:
                        prob_score = (clean_prob - patched_prob) / (clean_prob - corrupt_prob)
                    destruction_scores_prob[layer, head_idx, n_rep] = prob_score
                    clean_probs[layer, head_idx, n_rep] = clean_prob
                    corrupt_probs[layer, head_idx, n_rep] = corrupt_prob
                    patched_probs[layer, head_idx, n_rep] = patched_prob
                    
                    full_patched_probs = F.softmax(patched_logits[0, -1, :], dim=-1)
                    patched_winner[layer, head_idx, n_rep] = patched_prob == np.max(full_patched_probs.cpu().numpy())

    return destruction_scores_diff, destruction_scores_prob, clean_probs, corrupt_probs, patched_probs, patched_winner


# Define experiment parameters
prefix = "Jack went to the store.\nThe first time I was in the store, I"
base_sequence = " was in the store, and I was in the store."
clean_target = " I"
max_reps = 75
position = -1
knockout=True

# Needs to be quite long to cover max_reps of the sequence!
# E.g., for N=8, len = 8 * 5 tokens + 4 tokens = 44 tokens. 
# Let's provide a Wikipedia snippet as our natural baseline.
wiki_text = (
    """On 11 April 1951, U.S. president Harry S. Truman relieved General of the Army Douglas MacArthur of his commands after MacArthur made public statements that contradicted the administration's policies. MacArthur was a popular hero of World War II who was then commander of United Nations Command forces fighting in the Korean War, and his relief remains a controversial topic in the field of civil-military relations. United States General of the Army Douglas MacArthur shakes hands with US president Harry Truman at the Wake Island Conference, seven months before Truman relieved MacArthur from command. MacArthur led the Allied forces in the Southwest Pacific during World War II, and after the war was in charge of the occupation of Japan. In the latter role, MacArthur was able to accumulate considerable power over the civil administration of Japan. Eventually, he gained a level of political experience that was unprecedented and yet to be repeated by anyone else actively serving as a flag officer in the U.S. military. After North Korea invaded South Korea in June 1950, starting the Korean War, MacArthur was designated commander of the United Nations forces defending South Korea. He conceived and executed the amphibious assault at Inchon on 15 September 1950, but when he followed up his victory with a full-scale invasion of North Korea, China inflicted a series of defeats, compelling him to withdraw from North Korea. By April 1951, the military situation had stabilized, but MacArthur publicly criticized the administration's policies, leading Truman to have MacArthur relieved of his command. An apolitical military is an American tradition. The principle of civilian control of the military was also ingrained. Civilian control was an issue considering the constitutional division of powers between the president as commander-in-chief, and Congress with its power to raise armies, maintain a navy, and declare war. This was also an era when the rising complexity of military technology led to the creation of a professional military, and American forces were employed overseas in large numbers. The Armed Services Committee and the Foreign Relations Committee of the U.S. Senate held a joint inquiry into the military situation and the circumstances surrounding MacArthur's relief, and concluded that "the removal of General MacArthur was within the constitutional powers of the President but the circumstances were a shock to national pride".[1] In having MacArthur relieved for failing to "respect the authority of the President" by privately communicating with Congress, Truman upheld the president's role as preeminent. Harry S. Truman, then the vice president of the United States, became president of the United States on the death of Franklin D. Roosevelt in 1945, and won an unexpected victory to a full term in the 1948 presidential election. He was the only president who served after 1897 without a college degree. Although not highly educated, Truman was well read. When his high school friends went off to the state university in 1901, he enrolled in a local business school, but only lasted a semester. He later took night courses at the Kansas City Law School, but dropped out. Truman attempted to gain admission to the United States Military Academy at West Point, but was rejected for his poor eyesight. He was proud of his military service in the artillery during World War I, and continued to hold a reserve commission, eventually achieving the rank of colonel. Instead of professional soldiers, Truman selected two National Guardsmen, Harry H. Vaughan and Louis H. Renfrow, as his military aides. Truman once remarked that he did not understand how the US Army could "produce men such as Robert E. Lee, John J. Pershing, Eisenhower, and Bradley and at the same time produce Custers, Pattons, and MacArthur". During the 1949 Revolt of the Admirals, several naval officers publicly disagreed with the administration's policy over cuts to naval aviation and amphibious warfare capability, resulting in the relief of the Chief of Naval Operations, Admiral Louis Denfeld, and his replacement by Admiral Forrest Sherman. In testimony before the House Armed Services Committee investigation into the affair in October 1949, the chairman of the Joint Chiefs of Staff, General Omar Bradley, doubted that there would ever be another large-scale amphibious operation. In stature and seniority, General of the Army Douglas MacArthur was the Army's foremost general. The son of Lieutenant General Arthur MacArthur Jr., a recipient of the Medal of Honor for action during the American Civil War, he had graduated at the top of his West Point class of 1903, but never attended an advanced service school except for the engineer course in 1908. He had a distinguished combat record in World War I, and had served as Chief of Staff of the United States Army from 1930 to 1935, working closely with Presidents Herbert Hoover and Franklin Roosevelt, despite occasional clashes over the military budget. He would later compare Roosevelt's "extraordinary self-control" with Truman's "violent temper and paroxysms of ungovernable rage". Apart from his World War I-era service in Mexico and Europe, his overseas postings had been in Asia and the Pacific. During World War II, he had become a national hero and had been awarded the Medal of Honor for the unsuccessful defense of the Philippines in the Battle of Bataan. He had commanded the Allied armies in the New Guinea Campaign and Philippines Campaign, fulfilling his famous promise to return to the Philippines. In 1944 and 1948, he had been considered a possible Republican candidate for president. After the war, as the Supreme Commander of the Allied Powers, he had overseen the occupation of Japan and played an important part in the post-war political and social transformation of that country. By 1950, the occupation of Japan was winding down, but MacArthur remained in the country as Commander-in-Chief Far East, a post to which he had been appointed by Truman in 1945. MacArthur had to deal with deep cuts in the defense budget that saw his troop numbers decline from 300,000 in 1947 to 142,000 in 1948. Despite his protests, further reductions followed and, by June 1950, there were only 108,000 troops in his Far East Command. Cuts in funds and personnel produced shortages of serviceable equipment. Of the Far East Command's 18,000 jeeps, 10,000 were unserviceable; of its 13,780 2+1/2-ton 6x6 trucks, only 4,441 were serviceable. On the positive side, Far East Command initiated a program of reclaiming and refurbishing war materiel from abandoned stocks throughout the Pacific. This had not only recovered a great deal of valuable stores and equipment, it had also generated a useful repair and rebuilding industry in Japan. Meanwhile, the shift away from occupation duties had permitted a greater focus on training for combat. North Korea invaded South Korea on 25 June 1950, starting the Korean War. In response to an urgent request from the Korean Military Advisory Group for more ammunition, MacArthur, on his own initiative, ordered the transport ship MSTS Sgt. George D Keathley, then in harbor in Yokohama, to be loaded with ammunition and to sail for Pusan. President Truman met with the Joint Chiefs of Staff and other advisors that day at Blair House, his temporary residence during the White House Reconstruction, and approved the actions already taken by MacArthur and Secretary of State Dean Acheson. At another meeting at Blair House held on the evening of 26 June, amid reports of a rapidly deteriorating situation in South Korea, Truman approved the use of air and naval forces against military targets south of the 38th parallel north. Subsequently, on 27 June, the United Nations Security Council passed Resolution 83, which recommended that "members of the United Nations furnish such assistance to the Republic of Korea as may be necessary to repel the armed attack and to restore international peace and security in the area". The South Korean capital of Seoul fell on 28 June. The next day, Truman authorized air and naval operations north of the 38th parallel, which MacArthur had already ordered. However it was not until 30 June, following a sobering report on the military situation from MacArthur, that Truman finally authorized the use of ground forces. On 8 July, on the advice of the Joint Chiefs of Staff, Truman appointed MacArthur commander of United Nations Command in South Korea. He remained Commander-in-Chief Far East and Supreme Commander of the Allied Powers. MacArthur was forced to commit his forces in Japan to what he later described as a "desperate rearguard action". In July, Truman sent the Chief of Staff of the Army, General J. Lawton Collins, and the Chief of Staff of the Air Force, General Hoyt S. Vandenberg, to report on the situation. They met with MacArthur and his chief of staff, Major General Edward Almond, in Tokyo on 13 July. MacArthur impressed on them the danger of underestimating the North Koreans, whom he characterized as "well-equipped, well-led, and battle-trained, and which have at times out-numbered our troops by as much as twenty to one". He proposed to first halt the North Korean advance and then counterattack, enveloping the North Koreans with an amphibious operation, but the timing was dependent on the arrival of reinforcements from the United States. Bradley raised the possibility of using nuclear weapons in Korea at a Joint Chiefs of Staff meeting on 9 July 1950 at Eisenhower's instigation, but there was no support for the idea. The Army staff sent a cable to Collins in Tokyo suggesting that he seek out MacArthur's opinion. In a teleconference on 13 July, Major General Charles L. Bolte proposed sending nuclear weapons. MacArthur had already turned down Air Force proposals to firebomb North Korean cities, and suggested that atomic bombs could be used to isolate North Korea by taking out bridges and tunnels. The Army staff considered this impractical. On 28 July, the Joint Chiefs decided to send ten nuclear-capable B-29 bombers of the 9th Bombardment Wing to Guam as a deterrent to Chinese action against Taiwan. Truman publicly denied that he was considering the use of nuclear weapons in Korea, but authorized the transfer to Guam of atomic bombs without their fissile cores. The deployment did not go well; one of the bombers crashed on takeoff from Fairfield-Suisun Air Force Base in California on 5 August, killing the mission commander, Brigadier General Robert F. Travis, and 18 others. The remaining nine bombers remained in Guam until 13 September, when they returned to the United States. The bomb assemblies stayed behind. At a press conference on 13 July, Truman was asked if United States forces would cross the 38th parallel into North Korea, and he replied that he would "make that decision when it becomes necessary to do it". Some of his advisors, including the Assistant Secretary of State for Far Eastern Affairs, Dean Rusk, and the director of the Office of Northeast Asian Affairs, John M. Allison, argued that Security Council Resolution 83 provided a legal basis for the invasion of North Korea. Others, such as George F. Kennan and Paul Nitze, disagreed. Along with the legality, the administration also had to consider the danger of intervention by the Soviet Union or the People's Republic of China if United Nations forces approached their borders."""
)

print("Running natural text patching experiment...")
diff_matrix, prob_matrix, clean_probs_matrix, corrupt_probs_matrix, patched_probs_matrix, patched_winner_matrix = run_natural_neccesity_experiment(
    model, 
    base_seq=base_sequence, 
    clean_target=clean_target, 
    long_natural_text=wiki_text,
    max_repetitions=max_reps,
    prefix=prefix,
    position=position,
    knockout=knockout
)

In [ ]:
# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(diff_matrix[layer][i_head], "--.", label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Logits Destruction Score")
    plt.title("Logits Destruction Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(prob_matrix[layer][i_head], "--.", label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Probability Destruction Score")
    plt.title("Probability Destruction Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

In [ ]:
layer = 1
i_head = 3

plt.figure(figsize=(10,8))
plt.plot(clean_probs_matrix[layer][i_head], label=f"Clean Probs")
plt.plot(corrupt_probs_matrix[layer][i_head], label=f"Corrupt Probs")
plt.plot(patched_probs_matrix[layer][i_head], label=f"Patched Probs")
plt.plot(patched_winner_matrix[layer][i_head], "r.", label=f"Patched Won?")

plt.xlabel("# Repetitions")
# plt.ylabel("Logits Recovery Score [%]")
# plt.title("Logits Recovery Score vs. # Repetitions")
plt.legend()
plt.grid()

In [ ]:
from scipy.ndimage import uniform_filter1d as movmean

w = 7
plt.plot(movmean(diff_matrix[1][6], size=w, mode="nearest"))
plt.show()

In [ ]:
# Compute autocorrelation matrix of v_vectors for the selected head
v_head = v_vectors[:, head]  # shape: [sequence_length, d_head]

# Compute pairwise cosine similarity (autocorrelation)
autocorr_matrix = F.cosine_similarity(v_head.unsqueeze(1), v_head.unsqueeze(0), dim=-1)

# Plot as heatmap
plt.figure(figsize=(12, 10))
plt.imshow(autocorr_matrix.cpu().numpy(), cmap='coolwarm', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.xlabel('Position')
plt.ylabel('Position')
plt.title(f'Autocorrelation Matrix of V Vectors (head {head})')
plt.tight_layout()
plt.show()